## 1. Cargar datos

In [ ]:
import os
import pandas as pd

gold_standard = pd.read_csv(r'\datafiles\novelas_populares_lcquad_200_gold.csv')
model_output  = pd.read_csv(r'\Evaluation\\test_icl_main.csv')

print('gold_standard:', len(gold_standard), 'filas')
print('model_output:', len(model_output), 'filas')


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\natt\\datafiles\\novelas_populares_lcquad_200_gold.csv'

## 2. Corregir consultas SPARQL del modelo

El modelo generó URIs sin `< >`. Esta sección las corrige antes de re-ejecutar.

In [3]:
import re

# Patrón: URI http:// no precedida de < ' "
# Permite puntos internos (novelas-populares.org) pero no los captura al final
_URI_RE = re.compile(r'(?<![<\'"\\])https?://[^\s<>"\'{}()\[\]|]*(?<![.,;])')

def fix_bare_uris(query):
    if not isinstance(query, str):
        return query
    return _URI_RE.sub(r'<\g<0>>', query)

def clean_markdown_links(text):
    if not isinstance(text, str):
        return text
    return re.sub(r'\[([^\]]+)\]\(([^)]+)\)', r'\2', text)

def add_prefix_if_needed(query):
    if not isinstance(query, str):
        return query
    prefix = 'PREFIX np: <http://novelas-populares.org/>'
    if 'np:' in query and prefix not in query:
        return prefix + '\n' + query
    return query

def fix_query(query):
    query = clean_markdown_links(query)
    query = fix_bare_uris(query)
    query = add_prefix_if_needed(query)
    return query

model_output['consult_corr'] = model_output['consult'].apply(fix_query)

# Verificación rápida
for i, row in model_output.head(3).iterrows():
    print(f'--- {i} ---')
    print('Original: ', row['consult'][:100])
    print('Corregida:', row['consult_corr'][:100])
    print()


--- 0 ---
Original:  SELECT ?uri WHERE { ?uri http://novelas-populares.org/tieneTitulo "¡Huérfana!." . ?uri http://novela
Corregida: SELECT ?uri WHERE { ?uri <http://novelas-populares.org/tieneTitulo> "¡Huérfana!." . ?uri <http://nov

--- 1 ---
Original:  SELECT DISTINCT ?uri WHERE { http://novelas-populares.org/autor_emilio_salgari http://novelas-popula
Corregida: SELECT DISTINCT ?uri WHERE { <http://novelas-populares.org/autor_emilio_salgari> <http://novelas-pop

--- 2 ---
Original:  SELECT DISTINCT ?uri WHERE { http://novelas-populares.org/novela_culpable http://novelas-populares.o
Corregida: SELECT DISTINCT ?uri WHERE { <http://novelas-populares.org/novela_culpable> <http://novelas-populare



## 3. Re-ejecutar consultas contra Fuseki

In [4]:
import requests
from concurrent.futures import ThreadPoolExecutor

ENDPOINT = 'http://localhost:3030/KG_Novelas_Populares/sparql'
HEADERS  = {'Accept': 'application/sparql-results+json'}

def run_sparql(args):
    idx, query = args
    try:
        with requests.Session() as s:
            r = s.post(ENDPOINT, data={'query': query}, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return idx, r.json()
    except Exception as e:
        return idx, {'output error': str(e)}

N_WORKERS = 8
queries   = list(enumerate(model_output['consult_corr'].tolist()))
results   = [None] * len(queries)

print(f'Ejecutando {len(queries)} consultas ({N_WORKERS} hilos)...')
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    for idx, result in ex.map(run_sparql, queries):
        results[idx] = result

model_output['output'] = results

n_err = sum(isinstance(r, dict) and 'output error' in r for r in results)
print(f'Errores: {n_err} / {len(results)}')
if n_err > 0:
    sample = model_output[model_output['output'].apply(
        lambda x: isinstance(x, dict) and 'output error' in x)].head(3)
    for _, row in sample.iterrows():
        print(' ', row['consult_corr'][:120])
        print(' ', row['output'])


Ejecutando 360 consultas (8 hilos)...
Errores: 0 / 360


## 4. Preprocesar resultados

In [5]:
import ast

def safe_eval(x):
    if isinstance(x, (dict, bool)):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return x

def preprocess(output):
    """Devuelve set de URIs (SELECT) o bool (ASK). set() si hay error."""
    if not isinstance(output, dict) or 'output error' in output:
        return set()
    if 'results' in output:
        return {
            v['value']
            for row in output['results']['bindings']
            for v in row.values()
        }
    if 'boolean' in output:
        return output['boolean']
    return set()

def check_well_constructed(output):
    return not (isinstance(output, dict) and 'output error' in output)

def type_of_consult(consult):
    if not isinstance(consult, str): return 'NONE'
    if 'COUNT'    in consult: return 'COUNT'
    if 'ASK'      in consult: return 'ASK'
    if 'ORDER BY' in consult: return 'ORDER_BY'
    if 'SELECT'   in consult: return 'SELECT'
    return 'NONE'

model_output['evaluated']  = model_output['output'].apply(safe_eval)
gold_standard['evaluated'] = gold_standard['output'].apply(
    lambda x: safe_eval(x) if isinstance(x, str) else x)

model_output['curated']  = model_output['evaluated'].apply(preprocess)
gold_standard['curated'] = gold_standard['evaluated'].apply(preprocess)

model_output['model']  = 'gpt-4o-Mini'
model_output['method'] = model_output.get('method', pd.Series('main', index=model_output.index))
model_output['method'] = model_output['method'].fillna('main')


## 5. Evaluación

Se alinean model_output y gold_standard por la columna `query` / `corrected_question` para evitar errores por diferente orden de filas.

In [6]:
def compare_values(data1, data2):
    if isinstance(data1, set)  and isinstance(data2, set):  return data1 == data2
    if isinstance(data1, bool) and isinstance(data2, bool): return data1 == data2
    if type(data1) != type(data2): return False
    if isinstance(data1, dict):
        return data1.keys() == data2.keys() and all(
            compare_values(data1[k], data2[k]) for k in data1)
    if isinstance(data1, list):
        return sorted(str(x) for x in data1) == sorted(str(x) for x in data2)
    return data1 == data2

# Alinear por pregunta (no por posición)
merged = model_output.merge(
    gold_standard[['corrected_question', 'curated']],
    left_on='query', right_on='corrected_question',
    how='inner', suffixes=('_model', '_gold')
)
print(f'Preguntas alineadas: {len(merged)} de {len(model_output)}')

def eval_group(group):
    gold_slice  = merged.loc[group.index, 'curated_gold'].values
    model_slice = group['curated_model'].values
    evals = [compare_values(m, g) for m, g in zip(model_slice, gold_slice)]
    return pd.DataFrame({
        'evaluation': evals,
        'method':     group['method'].values,
        'consult':    group['consult_corr'].values,
        'output':     group['output'].values,
    })

grouped = merged.groupby('method', sort=False)
ex_eval = pd.concat([eval_group(g) for _, g in grouped], ignore_index=True)

ex_eval['evaluation'] = ex_eval['evaluation'].astype(int)
ex_eval['type']       = ex_eval['consult'].apply(type_of_consult)

for method, grp in ex_eval.groupby('method'):
    n_ok    = grp['evaluation'].sum()
    n_total = len(grp)
    n_valid = grp['output'].apply(check_well_constructed).sum()
    print(f'Método: {method} | Correctas: {n_ok}/{n_total} | Válidas: {n_valid}/{n_total}')


Preguntas alineadas: 360 de 360
Método: CoT | Correctas: 34/45 | Válidas: 45/45
Método: CoTont_rag | Correctas: 34/45 | Válidas: 45/45
Método: FS | Correctas: 40/45 | Válidas: 45/45
Método: FSCoT | Correctas: 34/45 | Válidas: 45/45
Método: FSCoTont_rag | Correctas: 34/45 | Válidas: 45/45
Método: FSont_rag | Correctas: 40/45 | Válidas: 45/45
Método: main | Correctas: 39/45 | Válidas: 45/45
Método: ont_rag | Correctas: 39/45 | Válidas: 45/45


## 6. Resultados

### Precision, Recall, F1, Jaccard

In [7]:
def metrics_for_pair(pred, gold):
    """
    Calcula precision, recall, F1 y Jaccard para un par (pred, gold).

    - Para resultados SELECT/COUNT: pred y gold son sets de URIs.
      Se tratan como conjuntos y se comparan elemento a elemento.
    - Para resultados ASK: pred y gold son bool.
      P = R = F1 = Jaccard = 1.0 si coinciden, 0.0 si no.
    - Si pred es set() por error de Fuseki: todas las métricas = 0.
    """
    # Caso ASK
    if isinstance(pred, bool) and isinstance(gold, bool):
        v = 1.0 if pred == gold else 0.0
        return {'precision': v, 'recall': v, 'f1': v, 'jaccard': v}

    # Caso SET (SELECT / COUNT)
    if not isinstance(pred, set): pred = set()
    if not isinstance(gold, set): gold = set()

    tp = len(pred & gold)
    fp = len(pred - gold)
    fn = len(gold - pred)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    jaccard   = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

    return {'precision': precision, 'recall': recall, 'f1': f1, 'jaccard': jaccard}

# Calcular métricas fila a fila sobre el merged
metric_rows = [
    metrics_for_pair(row['curated_model'], row['curated_gold'])
    for _, row in merged.iterrows()
]
metrics_df = pd.DataFrame(metric_rows)
metrics_df['method'] = merged['method'].values
metrics_df['type']   = merged['consult_corr'].apply(type_of_consult).values
metrics_df['model']  = 'gpt-4o-Mini'


In [8]:
# Resumen global por método
summary = (
    metrics_df.groupby('method')[['precision','recall','f1','jaccard']]
    .mean().round(4)
)
print(summary)


              precision  recall      f1  jaccard
method                                          
CoT              0.7556  0.7556  0.7556   0.7556
CoTont_rag       0.7556  0.7556  0.7556   0.7556
FS               0.8889  0.8889  0.8889   0.8889
FSCoT            0.7741  0.7778  0.7758   0.7741
FSCoTont_rag     0.7741  0.7778  0.7758   0.7741
FSont_rag        0.8889  0.8889  0.8889   0.8889
main             0.8667  0.8667  0.8667   0.8667
ont_rag          0.8667  0.8667  0.8667   0.8667


In [9]:
# Por método y tipo de consulta
summary_type = (
    metrics_df.groupby(['method','type'])[['precision','recall','f1','jaccard']]
    .mean().round(4)
)
print(summary_type)


                     precision  recall      f1  jaccard
method       type                                      
CoT          ASK        0.0000  0.0000  0.0000   0.0000
             COUNT      0.7778  0.7778  0.7778   0.7778
             SELECT     0.7714  0.7714  0.7714   0.7714
CoTont_rag   ASK        0.0000  0.0000  0.0000   0.0000
             COUNT      0.7778  0.7778  0.7778   0.7778
             SELECT     0.7714  0.7714  0.7714   0.7714
FS           ASK        0.7500  0.7500  0.7500   0.7500
             COUNT      0.8889  0.8889  0.8889   0.8889
             SELECT     0.9062  0.9062  0.9062   0.9062
FSCoT        ASK        0.0000  0.0000  0.0000   0.0000
             COUNT      0.8889  0.8889  0.8889   0.8889
             SELECT     0.7667  0.7714  0.7688   0.7667
FSCoTont_rag ASK        0.0000  0.0000  0.0000   0.0000
             COUNT      0.8889  0.8889  0.8889   0.8889
             SELECT     0.7667  0.7714  0.7688   0.7667
FSont_rag    ASK        0.7500  0.7500  0.7500  

In [10]:
# Tabla LaTeX completa
full_table = (
    metrics_df.groupby(['model','method','type'])[['precision','recall','f1','jaccard']]
    .mean() * 100
).round(2)
print(full_table.to_latex())


\begin{tabular}{lllrrrr}
\toprule
 &  &  & precision & recall & f1 & jaccard \\
model & method & type &  &  &  &  \\
\midrule
\multirow[t]{24}{*}{gpt-4o-Mini} & \multirow[t]{3}{*}{CoT} & ASK & 0.000000 & 0.000000 & 0.000000 & 0.000000 \\
 &  & COUNT & 77.780000 & 77.780000 & 77.780000 & 77.780000 \\
 &  & SELECT & 77.140000 & 77.140000 & 77.140000 & 77.140000 \\
\cline{2-7}
 & \multirow[t]{3}{*}{CoTont_rag} & ASK & 0.000000 & 0.000000 & 0.000000 & 0.000000 \\
 &  & COUNT & 77.780000 & 77.780000 & 77.780000 & 77.780000 \\
 &  & SELECT & 77.140000 & 77.140000 & 77.140000 & 77.140000 \\
\cline{2-7}
 & \multirow[t]{3}{*}{FS} & ASK & 75.000000 & 75.000000 & 75.000000 & 75.000000 \\
 &  & COUNT & 88.890000 & 88.890000 & 88.890000 & 88.890000 \\
 &  & SELECT & 90.620000 & 90.620000 & 90.620000 & 90.620000 \\
\cline{2-7}
 & \multirow[t]{3}{*}{FSCoT} & ASK & 0.000000 & 0.000000 & 0.000000 & 0.000000 \\
 &  & COUNT & 88.890000 & 88.890000 & 88.890000 & 88.890000 \\
 &  & SELECT & 76.670000 & 77.

In [11]:
# Accuracy global por método
ex_eval[['method','evaluation']].groupby('method').mean().round(4)


,evaluation
method,
CoT,0.7556
CoTont_rag,0.7556
FS,0.8889
FSCoT,0.7556
FSCoTont_rag,0.7556
FSont_rag,0.8889
main,0.8667
ont_rag,0.8667


In [12]:
# Accuracy por método y tipo de consulta
ex_eval[['method','evaluation','type']].groupby(['method','type']).mean().round(4)


evaluation
method       type              
CoT          ASK         0.0000
             COUNT       0.7778
             SELECT      0.7714
CoTont_rag   ASK         0.0000
             COUNT       0.7778
             SELECT      0.7714
FS           ASK         0.7500
             COUNT       0.8889
             SELECT      0.9062
FSCoT        ASK         0.0000
             COUNT       0.8889
             SELECT      0.7429
FSCoTont_rag ASK         0.0000
             COUNT       0.8889
             SELECT      0.7429
FSont_rag    ASK         0.7500
             COUNT       0.8889
             SELECT      0.9062
main         ASK         0.7500
             COUNT       0.7778
             SELECT      0.9062
ont_rag      ASK         0.7500
             COUNT       0.7778
             SELECT      0.9062

In [13]:
# Tabla final con modelo
ex_eval['model'] = 'gpt-4o-Mini'
output_table = (
    ex_eval[['model','method','evaluation','type']]
    .groupby(['model','method','type']).mean() * 100
).round(2)
print(output_table)


                                 evaluation
model       method       type              
gpt-4o-Mini CoT          ASK           0.00
                         COUNT        77.78
                         SELECT       77.14
            CoTont_rag   ASK           0.00
                         COUNT        77.78
                         SELECT       77.14
            FS           ASK          75.00
                         COUNT        88.89
                         SELECT       90.62
            FSCoT        ASK           0.00
                         COUNT        88.89
                         SELECT       74.29
            FSCoTont_rag ASK           0.00
                         COUNT        88.89
                         SELECT       74.29
            FSont_rag    ASK          75.00
                         COUNT        88.89
                         SELECT       90.62
            main         ASK          75.00
                         COUNT        77.78
                         SELECT 

In [14]:
print(output_table.to_latex())


\begin{tabular}{lllr}
\toprule
 &  &  & evaluation \\
model & method & type &  \\
\midrule
\multirow[t]{24}{*}{gpt-4o-Mini} & \multirow[t]{3}{*}{CoT} & ASK & 0.000000 \\
 &  & COUNT & 77.780000 \\
 &  & SELECT & 77.140000 \\
\cline{2-4}
 & \multirow[t]{3}{*}{CoTont_rag} & ASK & 0.000000 \\
 &  & COUNT & 77.780000 \\
 &  & SELECT & 77.140000 \\
\cline{2-4}
 & \multirow[t]{3}{*}{FS} & ASK & 75.000000 \\
 &  & COUNT & 88.890000 \\
 &  & SELECT & 90.620000 \\
\cline{2-4}
 & \multirow[t]{3}{*}{FSCoT} & ASK & 0.000000 \\
 &  & COUNT & 88.890000 \\
 &  & SELECT & 74.290000 \\
\cline{2-4}
 & \multirow[t]{3}{*}{FSCoTont_rag} & ASK & 0.000000 \\
 &  & COUNT & 88.890000 \\
 &  & SELECT & 74.290000 \\
\cline{2-4}
 & \multirow[t]{3}{*}{FSont_rag} & ASK & 75.000000 \\
 &  & COUNT & 88.890000 \\
 &  & SELECT & 90.620000 \\
\cline{2-4}
 & \multirow[t]{3}{*}{main} & ASK & 75.000000 \\
 &  & COUNT & 77.780000 \\
 &  & SELECT & 90.620000 \\
\cline{2-4}
 & \multirow[t]{3}{*}{ont_rag} & ASK & 75.000000 \\


In [15]:
ex_eval['valid'] = ex_eval['output'].apply(check_well_constructed)
print(ex_eval[['method','valid']].groupby('method').mean().to_latex())


\begin{tabular}{lr}
\toprule
 & valid \\
method &  \\
\midrule
CoT & 1.000000 \\
CoTont_rag & 1.000000 \\
FS & 1.000000 \\
FSCoT & 1.000000 \\
FSCoTont_rag & 1.000000 \\
FSont_rag & 1.000000 \\
main & 1.000000 \\
ont_rag & 1.000000 \\
\bottomrule
\end{tabular}



## 7. Depuración

In [16]:
# Comparaciones individuales tras el merge
for i, row in merged.head(10).iterrows():
    match = compare_values(row['curated_model'], row['curated_gold'])
    print(f'[{i}] {row["query"][:60]}')
    print(f'     model: {row["curated_model"]}')
    print(f'     gold : {row["curated_gold"]}')
    print(f'     match: {match}')
    print()


[0] ¿Cuál es el autor de ¡Huérfana!?
     model: set()
     gold : {'http://novelas-populares.org/autor_luis_de_val'}
     match: False

[1] ¿En qué país nació Emilio Salgari?
     model: {'http://novelas-populares.org/pais_italia'}
     gold : {'http://novelas-populares.org/pais_italia'}
     match: True

[2] ¿Qué editorial publicó ¿Culpable??
     model: set()
     gold : {'http://novelas-populares.org/editorial_e_domenech'}
     match: False

[3] ¿En qué país nació Arthur Conan Doyle?
     model: {'http://novelas-populares.org/pais_gran_bretana'}
     gold : {'http://novelas-populares.org/pais_gran_bretana'}
     match: True

[4] ¿Qué novelas publicó la Casa Editorial Seguí?
     model: {'http://novelas-populares.org/novela_1682494306', 'http://novelas-populares.org/novela_1682490300', 'http://novelas-populares.org/novela_1682518930', 'http://novelas-populares.org/novela_1682489914', 'http://novelas-populares.org/novela_1682099695', 'http://novelas-populares.org/novela_1682381854', 

In [17]:
# Consultas con error
errored = model_output[model_output['output'].apply(
    lambda x: isinstance(x, dict) and 'output error' in x)]
print(f'{len(errored)} consultas con error:')
for _, row in errored.head(5).iterrows():
    print(' ', row['consult_corr'][:120])
    print(' ', row['output'])
    print()


0 consultas con error:
